# 04 · Our contribution: fine-tuning Whisper-small and wav2vec 2.0 XLS-R on our train split

Runs on Kaggle with a GPU. **Settings → Accelerator: GPU, Internet: On.** No manual uploads.
Set `SYSTEM` to `whisper_small_ft` or `xlsr_300m_ft` (one Kaggle session each).

`scripts/finetune.py` trains on the train split, chooses the checkpoint and the decoding settings
on dev, and decodes test once at the end (settings: `configs/finetune.yaml`).
`TRIAL = True` runs 30 steps on 200 train clips and 20 dev/test clips into `/kaggle/working/trial`.

Output: `/kaggle/working/results_<SYSTEM>.zip` (predictions, `.meta.json`, training log) and the
fine-tuned weights `/kaggle/working/<SYSTEM>_best/` (not in the repository).

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
SYSTEM = "whisper_small_ft"
TRIAL = False

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; using anonymous access")

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q -b $BRANCH $REPO_URL repo
%cd /kaggle/working/repo
!git log -1 --format='commit %H'
!pip install -q -r requirements/finetune.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze | grep -iE '^(torch|transformers|huggingface[_-]hub|numpy|jiwer|pyyaml)=='

In [ ]:
!python -m pytest -q tests/test_finetune.py tests/test_inference.py

In [ ]:
!python scripts/unpack_audio.py

In [ ]:
import subprocess, sys
extra = ["--max-steps", "30", "--limit-train", "200", "--limit-eval", "20", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
r = subprocess.run([sys.executable, "scripts/finetune.py", "--system", SYSTEM, *extra])
print("exit code", r.returncode)

In [ ]:
if TRIAL:
    !head -c 3000 /kaggle/working/trial/train_logs/$SYSTEM.json
    !head -n 3 /kaggle/working/trial/predictions/$SYSTEM.dev.jsonl
else:
    !cp -r work/models/$SYSTEM/best /kaggle/working/{SYSTEM}_best
    !zip -q -r /kaggle/working/results_{SYSTEM}.zip results/predictions/{SYSTEM}.* results/train_logs/{SYSTEM}.json
    !ls -lh /kaggle/working